# Power BI Queries

Sample SQL queries for Power BI report building. Copy these directly into Power BI as SQL Server data sources.

**Setup in Power BI:**
1. Get Data → SQL Server
2. Server: `<your-workspace>.dfs.core.windows.net` (OneLake)
3. Database: (leave empty)
4. Copy query below into "Advanced options" → SQL statement
5. Create relationships between these tables

Use these queries as the foundation for dashboards, trend analysis, and failure tracking.

In [ ]:
# -- Configuration -------------------------------------------------------------
# Load shared settings from config.py in Lakehouse.
import sys
sys.path.append("/lakehouse/default/Files/code")

try:
    from config import LAKEHOUSE_NAME, SCHEMA_NAME, MAX_RETRY_ATTEMPTS, INITIAL_RETRY_DELAY, DAX_TIMEOUT_SECONDS, RUN_TABLE_MAINTENANCE, MAINTENANCE_DAY_UTC
except ImportError:
    # Fallback keeps the notebook runnable if shared config execution is unavailable.
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"
    MAX_RETRY_ATTEMPTS = 3
    INITIAL_RETRY_DELAY = 2
    DAX_TIMEOUT_SECONDS = 60
    RUN_TABLE_MAINTENANCE = False
    MAINTENANCE_DAY_UTC = 6

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"

print(f"Schema: {FULL_SCHEMA}")

## Dimension Tables

Use these for filtering and grouping in Power BI.

### Dim_Models

All semantic models in the validation system.

In [ ]:
sql_dim_models = f"""
WITH model_sources AS (
    SELECT model_name, workspace_id, dataset_id, workspace_name, dataset_name
    FROM {FULL_SCHEMA}.check_registry

    UNION ALL

    SELECT DISTINCT model_name, workspace_id, dataset_id, workspace_name, dataset_name
    FROM {FULL_SCHEMA}.validation_results
)
SELECT
    model_name,
    workspace_id,
    dataset_id,
    MAX(workspace_name) AS workspace_name,
    MAX(dataset_name) AS dataset_name
FROM model_sources
GROUP BY model_name, workspace_id, dataset_id
ORDER BY model_name, workspace_id, dataset_id
"""

print("Copy this query into Power BI:")
print(sql_dim_models)

### Dim_Checks

All registered metric checks.

In [ ]:
sql_dim_checks = f"""
WITH check_sources AS (
    SELECT check_name, model_name, run_frequency
    FROM {FULL_SCHEMA}.check_registry

    UNION ALL

    SELECT DISTINCT check_name, model_name, CAST(NULL AS STRING) AS run_frequency
    FROM {FULL_SCHEMA}.validation_results
)
SELECT
    check_name,
    COUNT(DISTINCT model_name) AS model_count,
    MAX(CASE WHEN run_frequency = 'MULTIPLE_PER_DAY' THEN 1 ELSE 0 END) AS has_multiple_per_day,
    CASE
        WHEN MAX(CASE WHEN run_frequency = 'MULTIPLE_PER_DAY' THEN 1 ELSE 0 END) = 1
            THEN 'MULTIPLE_PER_DAY'
        WHEN MAX(CASE WHEN run_frequency = 'ONCE_PER_DAY' THEN 1 ELSE 0 END) = 1
            THEN 'ONCE_PER_DAY'
        ELSE 'UNKNOWN_HISTORICAL'
    END AS run_frequency
FROM check_sources
GROUP BY check_name
ORDER BY check_name
"""

print("Copy this query into Power BI:")
print(sql_dim_checks)

### Dim_Date

All dates with validation results (for time-based filtering).

In [ ]:
sql_dim_date = f"""
SELECT DISTINCT
    run_date AS date_key,
    run_date,
    YEAR(run_date) AS year,
    MONTH(run_date) AS month,
    DAY(run_date) AS day,
    DAYNAME(run_date) AS day_name,
    DAYOFWEEK(run_date) AS day_of_week,
    WEEKOFYEAR(run_date) AS week_of_year,
    DATE_TRUNC('MONTH', run_date) AS month_start
FROM {FULL_SCHEMA}.validation_results
ORDER BY run_date DESC
"""

print("Copy this query into Power BI:")
print(sql_dim_date)

## Fact Tables

Use these for detailed analysis and calculations in Power BI.

### Fact_ValidationResults

All validation results with calculations for Power BI.

In [ ]:
sql_fact_validation_results = f"""
SELECT
    run_id,
    run_date,
    run_timestamp,
    check_name,
    model_name,
    workspace_id,
    dataset_id,
    workspace_name,
    dataset_name,
    result_value,
    baseline_model,
    baseline_value,
    absolute_delta,
    delta_pct,
    fail_delta_pct_threshold,
    status,
    error_message,
    CASE 
        WHEN status = 'PASS' THEN 1 
        ELSE 0 
    END AS is_pass,
    CASE 
        WHEN status = 'FAIL' THEN 1 
        ELSE 0 
    END AS is_fail,
    CASE 
        WHEN status = 'ERROR' THEN 1 
        ELSE 0 
    END AS is_error,
    ABS(COALESCE(delta_pct, 0)) AS abs_delta_pct
FROM {FULL_SCHEMA}.validation_results
ORDER BY run_date DESC, check_name, model_name
"""

print("Copy this query into Power BI:")
print(sql_fact_validation_results)

### Fact_Failures

Subset of validation results where checks failed or errored. Use for failure dashboard.


In [ ]:
sql_fact_failures = f"""
SELECT
    run_id,
    run_date,
    run_timestamp,
    check_name,
    model_name,
    workspace_id,
    dataset_id,
    workspace_name,
    dataset_name,
    result_value,
    baseline_model,
    baseline_value,
    absolute_delta,
    delta_pct,
    fail_delta_pct_threshold,
    status,
    error_message
FROM {FULL_SCHEMA}.validation_results
WHERE status IN ('FAIL', 'ERROR')
ORDER BY run_date DESC, check_name
"""

print("Copy this query into Power BI for failure dashboard:")
print(sql_fact_failures)

### Dim_ModelRoles

Role dimension sourced from fact behavior (baseline vs comparison).

In [ ]:
sql_dim_model_roles = f"""
WITH model_keys AS (
    SELECT DISTINCT check_name, model_name, workspace_id, dataset_id
    FROM {FULL_SCHEMA}.check_registry

    UNION

    SELECT DISTINCT check_name, model_name, workspace_id, dataset_id
    FROM {FULL_SCHEMA}.validation_results
),
baseline_cfg AS (
    SELECT
        check_name,
        COALESCE(baseline_mode, 'MODEL') AS baseline_mode,
        static_baseline_value
    FROM {FULL_SCHEMA}.check_baseline_config
),
registry_baselines AS (
    SELECT
        check_name,
        model_name AS baseline_model,
        workspace_id AS baseline_workspace_id,
        dataset_id AS baseline_dataset_id
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
      AND is_baseline = true
)
SELECT
    k.check_name,
    k.model_name,
    k.workspace_id,
    k.dataset_id,
    CASE
        WHEN UPPER(COALESCE(cfg.baseline_mode, 'MODEL')) = 'STATIC' THEN 'Comparison'
        WHEN k.model_name = r.baseline_model
         AND k.workspace_id = r.baseline_workspace_id
         AND k.dataset_id = r.baseline_dataset_id THEN 'Baseline'
        ELSE 'Comparison'
    END AS model_role,
    COALESCE(cfg.baseline_mode, 'MODEL') AS baseline_mode,
    cfg.static_baseline_value
FROM model_keys k
LEFT JOIN baseline_cfg cfg
    ON k.check_name = cfg.check_name
LEFT JOIN registry_baselines r
    ON k.check_name = r.check_name
ORDER BY k.check_name, k.model_name, k.workspace_id, k.dataset_id
"""

print("Copy this query into Power BI:")
print(sql_dim_model_roles)

sql_fact_trends = f"""
SELECT
    run_date,
    check_name,
    COUNT(*) as total_checks,
    SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) as pass_count,
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) as fail_count,
    SUM(CASE WHEN status = 'ERROR' THEN 1 ELSE 0 END) as error_count,
    ROUND(
        100.0 * SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) as pass_rate_pct
FROM {FULL_SCHEMA}.validation_results
GROUP BY run_date, check_name
ORDER BY run_date DESC, check_name
"""

print("\nCopy this query into Power BI for trend analysis:")
print(sql_fact_trends)

## How to Use in Power BI

### Step 1: Connect to OneLake
1. Open Power BI Desktop
2. Home → Get Data → Azure → OneLake
3. Select your Fabric workspace and lakehouse
4. Navigate to `{SCHEMA_NAME}` → `validation_results`

### Step 2: Load Query Tables

For each fact/dimension table above:
1. Home → New Source → SQL → Enter the SQL query
2. Click Load to import the data
3. Repeat for all 7 tables (4 dimensions + 3 facts)

### Step 3: Create Relationships

In Power BI Data Model:
- **Dim_Checks** → **Fact_ValidationResults** on `check_name` (one-to-many)
- **Dim_Checks** → **Fact_Failures** on `check_name` (one-to-many)
- **Dim_Checks** → **Fact_Trends** on `check_name` (one-to-many)
- **Dim_Date** → **Fact_Trends** on `run_date` (one-to-many)
- **Dim_Models** → **Fact_ValidationResults** on composite key: `model_name` + `workspace_id` + `dataset_id`
- **Dim_Models** → **Fact_Failures** on composite key: `model_name` + `workspace_id` + `dataset_id`
- **Dim_ModelRoles** → **Fact_ValidationResults** on composite key: `check_name` + `model_name` + `workspace_id` + `dataset_id`
- **Dim_ModelRoles** → **Fact_Failures** on composite key: `check_name` + `model_name` + `workspace_id` + `dataset_id`

### Step 4: Build Visualizations

**Suggested Pages:**
- **Health Dashboard:** Fact_Trends pass_rate_pct card, line chart by run_date
- **Failure Detail:** Fact_Failures table, filtered to status='FAIL'
- **Model Comparison:** Fact_ValidationResults matrix: check_name × model_name → pass_count
- **Trend Analysis:** Line chart: run_date × pass_rate_pct, sliced by Dim_Checks